## Vectors, Embeddings, Semantic Search and RAG

In [18]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-chroma \
    langchain-text-splitters \
    langchain-classic \
    pypdf \
    chromadb \
    python-dotenv \
    pandas \
    numpy \
    tiktoken

### OpenAI embeddings capture semantic meaning

In [5]:
import os
import numpy as np
import pandas as pd
from typing import List, Optional, Dict

from dotenv import load_dotenv
from openai import OpenAI

In [6]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully")

OpenAI client initialized successfully


In [7]:
vector_a = np.array([1, 2])
vector_b = np.array([2, 4])

print("Vector A:", vector_a)
print("Vector B:", vector_b)

Vector A: [1 2]
Vector B: [2 4]


## Create Cosine Similarity Function

In [8]:
def cosine_similarity(a, b):
    dot_product = np.dot(a, b)
    magnitude_a = np.linalg.norm(a)
    magnitude_b = np.linalg.norm(b)

    return dot_product / (magnitude_a * magnitude_b)

In [9]:
score = cosine_similarity(vector_a, vector_b)

print("Cosine similarity between Vector A and Vector B:", score)

Cosine similarity between Vector A and Vector B: 0.9999999999999998


## Convert Text into Embeddings

In [10]:
def generate_embedding(text: str, model: str = "text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )

    return response.data[0].embedding

## Generate Embedding for One Sentence

In [11]:
text = "OpenAI embeddings capture semantic meaning for search applications"

embedding = generate_embedding(text)

print("Embedding type:", type(embedding))

print("Number of dimensions in embedding:", len(embedding))

print("First 10 values of the embedding:", embedding[:10])


Embedding type: <class 'list'>
Number of dimensions in embedding: 1536
First 10 values of the embedding: [-0.0133209228515625, -0.018890380859375, 0.0246429443359375, -0.004329681396484375, 0.0005855560302734375, -0.0306243896484375, -0.0160064697265625, 0.0224151611328125, 0.0237274169921875, -0.006580352783203125]


## Compare Two Similar Sentences

In [12]:
sentence_1 = "How to measure search quality?"
sentence_2 = "Retrieval evaluation requires precision and recall metrics"

embedding_1 = generate_embedding(sentence_1)
embedding_2 = generate_embedding(sentence_2)

similarity = cosine_similarity(embedding_1, embedding_2)

print("Similarity Score:", similarity)

Similarity Score: 0.44475398151912604


## Manual Semantic Search

PDF files
Word documents
Web pages
CSV files
Knowledge base articles
Company policies
Book chapters

In [13]:
documents = [
    "ChromaDB supports metadata filtering and CRUD operations",
    "OpenAI embeddings capture semantic meaning for search applications",
    "HNSW indexing enables efficient approximate nearest neighbor search",
    "Retrieval evaluation requires precision@k and recall@k metrics",
    "Vector databases store embeddings for similarity search"
]

In [14]:
document_embeddings = []

for doc in documents:
    emb = generate_embedding(doc)
    document_embeddings.append(emb)


print("Total documents embedded:", len(document_embeddings))
print("Embedding size of first document:", len(document_embeddings[0]))

Total documents embedded: 5
Embedding size of first document: 1536


## Search Manually Using Cosine similarity

1. Takes user query.
2. Converts query into embedding.
3. Compares query embedding with every document embedding.
4. Calculates similarity score.
5. Returns top matching documents.

In [15]:
def manual_semantic_search(query: str, top_k: int = 3):
    query_embedding = generate_embedding(query)

    results = []

    for index, doc in enumerate(documents):
        score = cosine_similarity(query_embedding, document_embeddings[index])

        results.append({
            "document": doc,
            "similarity_score": score
        })

    results_df   = pd.DataFrame(results)
    results_df.sort_values(by="similarity_score", ascending=False, inplace=True)
    return results_df.head(top_k)

In [16]:
manual_semantic_search("How do I measure retrieval quality?")

,document,similarity_score
3,Retrieval evaluation requires precision@k and ...,0.650445
4,Vector databases store embeddings for similari...,0.262979
1,OpenAI embeddings capture semantic meaning for...,0.250455


## Vector Databases- 

- ChromaDB
- Pinecone
- Weaviate
- FAISS
- Milvus
- Qdrant

In [19]:
import chromadb

In [20]:
class OpenAIEmbeddingFunction:
    def __init__(self, model_name: str = "text-embedding-3-small"):
        self.model_name = model_name

    def __call__(self, input: List[str]) -> List[List[float]]:
        response = client.embeddings.create(
            model=self.model_name,
            input=input
        )

        return [item.embedding for item in response.data]

    def name(self) -> str:
        return "openai-embedding-function"

## Create Persistent ChromaDB Client

In [21]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

openai_ef = OpenAIEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="tech_docs",
    embedding_function=openai_ef
)

print("ChromaDB collection created successfully.")

ChromaDB collection created successfully.


## Add Documents to ChromaDB

In [22]:
metadatas = [
    {"category": "chromadb", "source": "docs"},
    {"category": "embeddings", "source": "research"},
    {"category": "indexing", "source": "paper"},
    {"category": "evaluation", "source": "tutorial"},
    {"category": "vector-db", "source": "textbook"}
]

ids = [f"doc{i+1}" for i in range(len(documents))]

In [23]:
collection.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

## Semantic Search Using ChromaDB

In [24]:
def search_pipeline(query: str, n_results: int = 3, filters: Optional[Dict] = None) -> pd.DataFrame:
    query_embedding = openai_ef([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=filters
    )

    return pd.DataFrame({
        "Document": results["documents"][0],
        "ID": results["ids"][0],
        "Distance": results["distances"][0],
        "Metadata": results["metadatas"][0]
    })

In [25]:
results_df = search_pipeline("How do I measure retrieval quality?", n_results=3)

print(results_df)

                                            Document    ID  Distance  \
0  Retrieval evaluation requires precision@k and ...  doc4  0.699078   
1  Vector databases store embeddings for similari...  doc5  1.474373   
2  OpenAI embeddings capture semantic meaning for...  doc2  1.499109   

                                           Metadata  
0  {'category': 'evaluation', 'source': 'tutorial'}  
1   {'source': 'textbook', 'category': 'vector-db'}  
2  {'source': 'research', 'category': 'embeddings'}  


In [26]:
filtered_results = search_pipeline(
    query="What is HNSW indexing?",
    n_results=2,
    filters={"category": "indexing"}
)

print(filtered_results)

                                            Document    ID  Distance  \
0  HNSW indexing enables efficient approximate ne...  doc3  0.504093   

                                      Metadata  
0  {'category': 'indexing', 'source': 'paper'}  


## Evaluation of Search Quality

Retrieval Evaluation Matrics

- precision@k --> Out of top k results, How many are correct?

Top 2 results retured
2 are correct
Precision@3 = 2/3 = 0.66
High precision means clean results.


- Recall@k --> Out of all correct documents, How many did we retrieve

Total correct documents = 4
Retrieved correct documents = 2
Recall@k = 2/4 = 0.5
High recall means fewer important documents are missed.


- MRR --> Mean Reciprocal Rank --> At what rank  did the first correct answer appear?

MRR = 1/1 = 1.0
MRR = 1/2 = 0.5
MRR = 1/3 = 0.33

High MRR means the correct answer appears early.

## Ground Truth Data

In [27]:
test_queries = [
    {
        "question": "ChromaDB Features",
        "correct_document_id": "doc1"
    },
    {
        "question": "Similarity search database",
        "correct_document_id": "doc5"
    },
    {
        "question": "Evaluation metrics for retrieval",
        "correct_document_id": "doc4"
    }
]

## Chceck the Seach Result for one Question

In [28]:
questions = "ChromaDB Features"

search_result = search_pipeline(
    query = questions, 
    n_results=2
    
)

search_result

,Document,ID,Distance,Metadata
0,ChromaDB supports metadata filtering and CRUD ...,doc1,0.428855,"{'source': 'docs', 'category': 'chromadb'}"
1,Vector databases store embeddings for similari...,doc5,1.429028,"{'category': 'vector-db', 'source': 'textbook'}"


## Extract only Retrieved IDs

In [29]:
retrieved_ids = search_result["ID"].to_list()

print("Retrieved Document IDs:", retrieved_ids)

Retrieved Document IDs: ['doc1', 'doc5']


## Check whether correct Document was found

In [30]:
Correct_document_id = retrieved_ids[0]  # Assuming the top result should be the correct document

if Correct_document_id in retrieved_ids:
    print(f"Test passed: Correct document ID '{Correct_document_id}' was retrieved.")
else:
    print(f"Test failed: Correct document ID '{Correct_document_id}' was NOT retrieved.")    

Test passed: Correct document ID 'doc1' was retrieved.


## Understanding Precision@k in simple terms

In [31]:
#calculate accuracy using Precision@k

k =2

correct_count = 0
for doc_id in retrieved_ids:
    if doc_id == Correct_document_id:
        correct_count += 1


precision = correct_count / k

print("Correct count:", correct_count)
print(f"Precision@{k}: {precision}")

Correct count: 1
Precision@2: 0.5


 Understanding Recall@k in simple terms

In [32]:
total_correct_documents = 1  # We have one correct document in our test set

recall = correct_count / total_correct_documents

print(f"Recall@{k}: {recall}")

Recall@2: 1.0


## Understanding MRR

In [33]:
mrr = 0

for position, doc_id in enumerate(retrieved_ids, start=1):
    if doc_id == Correct_document_id:
        mrr = 1 / position
        break

print("MRR:", mrr)

MRR: 1.0
